# 03 — Calibration & Explainability

**Research use only. Not for clinical decisions.**

Set `DATA_PATH`, `TARGET_COLUMN`, and `MODEL_PATH` before running.

In [ ]:
import os, sys
sys.path.insert(0, '../src')
DATA_PATH = os.environ.get('DATA_PATH', '')
TARGET_COLUMN = os.environ.get('TARGET_COLUMN', 'severe')
MODEL_PATH = os.environ.get('MODEL_PATH', 'outputs/demo/best_model.joblib')
if not DATA_PATH:
    raise RuntimeError('Set DATA_PATH. No dataset is bundled.')

In [ ]:
import joblib
from penux_ap.datasets import load_dataset
from penux_ap.labels import binarize_target
from penux_ap.models import predict_proba_safe
from penux_ap.calibration import calibration_curve_data, brier_score
from penux_ap.explainability import permutation_importance_report, shap_report_if_available
from penux_ap.evaluation import confusion_matrix_at_thresholds

df = load_dataset(DATA_PATH)
y = binarize_target(df[TARGET_COLUMN]).dropna().astype(int)
X = df.drop(columns=[TARGET_COLUMN]).loc[y.index]
model = joblib.load(MODEL_PATH)
y_proba = predict_proba_safe(model, X)

In [ ]:
cal_data = calibration_curve_data(y.values, y_proba)
print('Brier score:', brier_score(y.values, y_proba))
print('Calibration data:', cal_data)

In [ ]:
# Confusion matrices at multiple thresholds
cms = confusion_matrix_at_thresholds(y.values, y_proba)
for cm in cms:
    print(f"Threshold {cm['threshold']}: TP={cm['TP']} TN={cm['TN']} FP={cm['FP']} FN={cm['FN']}")

In [ ]:
fi = permutation_importance_report(model, X, y)
print(fi.head(10))

shap_result = shap_report_if_available(model, X)
if shap_result:
    print('SHAP mean abs:', shap_result)